<a href="https://colab.research.google.com/github/NandoRoG/Specscript_a_DSL_for_Batch_Speculative_Decoding/blob/main/SpectScript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q transformers accelerate

In [3]:
# ── specscript/ir.py ─────────────────────────────────────────
# Typed IR that representsany batch speculative decoding strategy as a graph of nodes.

from dataclasses import dataclass, field
from typing import List, Optional
from enum import Enum, auto

class NodeKind(Enum):
    DRAFT = auto()
    VERIFY = auto()
    ACCEPT = auto()
    REALIGN = auto()
    POOL = auto()

@dataclass
class SpecNode:
    kind: NodeKind
    node_id: int = 0
    inputs: List[int] = field(default_factory=list)
    outputs: List[int] = field(default_factory=list)

@dataclass
class DraftNode(SpecNode):
    """Runs draft model to propose k tokens per sequence."""
    kind: NodeKind = field(default=NodeKind.DRAFT, init=False)
    draft_model_ref: str = ""
    num_draft_tokens: int = 5

@dataclass
class VerifyNode(SpecNode):
    """Runs target model over (sequence + draft) in one forward pass."""
    kind: NodeKind = field(default=NodeKind.VERIFY, init=False)
    target_model_ref: str = ""

@dataclass
class AcceptNode(SpecNode):
    """
    Accepts the longest correct draft prefix.
    INVARIANT: bonus token MUST come from target model, never draft.
    This is the bug in DSD, BSP, and Meta's implementation.
    """
    kind: NodeKind = field(default=NodeKind.ACCEPT, init=False)
    sampling: str = "greedy"

@dataclass
class RealignNode(SpecNode):
    """
    Enforces three synchronization invariants (Zhang et al., 2026):
      INV-1 (I1): p_i + c_i = L  for all i  (rectangular alignment)
      INV-2 (I2): position IDs are contiguous, reset to 0 at first content token
      INV-3    : KV-cache shifted by delta_i to match new padding offsets
    """
    kind: NodeKind = field(default=NodeKind.REALIGN, init=False)

@dataclass
class PoolNode(SpecNode):
    """EXSPEC variant: group same-length sequences, defer realignment."""
    kind: NodeKind = field(default=NodeKind.POOL, init=False)
    window_size: int = 16

@dataclass
class SpecGraph:
    nodes: List[SpecNode] = field(default_factory=list)
    _next_id: int = field(default=0, repr=False)

    def add(self, node: SpecNode) -> SpecNode:
        node.node_id = self._next_id
        self._next_id += 1
        self.nodes.append(node)
        return node

print("✅ SpecScript IR loaded — 5 node types, SpecGraph container")
print(f"   Node kinds: {[k.name for k in NodeKind]}")

✅ SpecScript IR loaded — 5 node types, SpecGraph container
   Node kinds: ['DRAFT', 'VERIFY', 'ACCEPT', 'REALIGN', 'POOL']


In [4]:
# ── specscript/checker.py ────────────────────────────────────
# Walks the IR graph BEFORE execution and rejects strategies
# that violate the synchronization invariants.
# This is what none of BSP, DSD, or Meta's system had.

class SpecScriptError(Exception):
    pass

def check_strategy(graph: SpecGraph) -> None:
    """
    Static correctness checker. Raises SpecScriptError if the
    strategy violates any synchronization invariant.

    Rules enforced:
      R1: Every AcceptNode must be immediately followed by a
          RealignNode or PoolNode  (enforces INV-1, INV-2, INV-3)
      R2: Strategy must begin with a DraftNode
      R3: Strategy must contain at least one VerifyNode
      R4: A VerifyNode must follow a DraftNode (no verify without draft)
    """
    kinds = [n.kind for n in graph.nodes]

    # R2: must start with Draft
    if not kinds or kinds[0] != NodeKind.DRAFT:
        raise SpecScriptError(
            "[R2] Strategy must begin with a DraftNode. "
            "Without a draft model, there are no tokens to verify."
        )

    # R3: must have at least one Verify
    if NodeKind.VERIFY not in kinds:
        raise SpecScriptError(
            "[R3] Strategy has no VerifyNode. "
            "A target model forward pass is required for output equivalence."
        )

    # R4: Verify must follow Draft
    for i, node in enumerate(graph.nodes):
        if node.kind == NodeKind.VERIFY:
            if i == 0 or graph.nodes[i-1].kind != NodeKind.DRAFT:
                raise SpecScriptError(
                    f"[R4] VerifyNode at position {i} is not preceded by a DraftNode. "
                    "Verification requires draft tokens to check."
                )

    # R1: Every Accept must be followed by Realign or Pool
    for i, node in enumerate(graph.nodes):
        if node.kind == NodeKind.ACCEPT:
            if i + 1 >= len(graph.nodes):
                raise SpecScriptError(
                    f"[R1] AcceptNode at position {i} is the last node — "
                    "no RealignNode follows it. This violates INV-1 (rectangular alignment), "
                    "INV-2 (position-ID contiguity), and INV-3 (KV-cache correspondence). "
                    "This is the exact bug class found in BSP, DSD, and Meta's system."
                )
            next_kind = graph.nodes[i+1].kind
            if next_kind not in (NodeKind.REALIGN, NodeKind.POOL):
                raise SpecScriptError(
                    f"[R1] AcceptNode at position {i} is followed by {next_kind.name}, "
                    f"not a RealignNode or PoolNode. "
                    "Without realignment, position IDs desynchronize and KV-cache "
                    "entries point to wrong token positions — silent output corruption."
                )

    print("✅ Static check PASSED — strategy satisfies all synchronization invariants")


# ─── Test 1: VALID strategy (EQSPEC pattern) ─────────────────
print("=" * 55)
print("TEST 1: Valid EQSPEC strategy")
g_valid = SpecGraph()
g_valid.add(DraftNode(node_id=0, draft_model_ref="opt-125m", num_draft_tokens=5))
g_valid.add(VerifyNode(node_id=0, target_model_ref="opt-1.3b", inputs=[0]))
g_valid.add(AcceptNode(node_id=0, inputs=[1]))
g_valid.add(RealignNode(node_id=0, inputs=[2]))
check_strategy(g_valid)
print(f"Graph: {' → '.join(n.kind.name for n in g_valid.nodes)}")

# ─── Test 2: INVALID strategy (missing Realign — the DSD bug) ─
print()
print("TEST 2: Broken strategy — Accept with no Realign (DSD/BSP bug)")
g_broken = SpecGraph()
g_broken.add(DraftNode(node_id=0, draft_model_ref="opt-125m"))
g_broken.add(VerifyNode(node_id=0, target_model_ref="opt-1.3b", inputs=[0]))
g_broken.add(AcceptNode(node_id=0, inputs=[1]))
# ← deliberately missing RealignNode
try:
    check_strategy(g_broken)
except SpecScriptError as e:
    print(f"🚫 Caught before execution: {e}")

# ─── Test 3: INVALID strategy (no draft) ─────────────────────
print()
print("TEST 3: Broken strategy — starts with Verify (no Draft)")
g_nodraft = SpecGraph()
g_nodraft.add(VerifyNode(node_id=0, target_model_ref="opt-1.3b"))
g_nodraft.add(AcceptNode(node_id=0, inputs=[0]))
g_nodraft.add(RealignNode(node_id=0, inputs=[1]))
try:
    check_strategy(g_nodraft)
except SpecScriptError as e:
    print(f"🚫 Caught before execution: {e}")

print()
print("=" * 55)
print("Static checker: 1 valid strategy passed, 2 broken strategies caught")

TEST 1: Valid EQSPEC strategy
✅ Static check PASSED — strategy satisfies all synchronization invariants
Graph: DRAFT → VERIFY → ACCEPT → REALIGN

TEST 2: Broken strategy — Accept with no Realign (DSD/BSP bug)
🚫 Caught before execution: [R1] AcceptNode at position 2 is the last node — no RealignNode follows it. This violates INV-1 (rectangular alignment), INV-2 (position-ID contiguity), and INV-3 (KV-cache correspondence). This is the exact bug class found in BSP, DSD, and Meta's system.

TEST 3: Broken strategy — starts with Verify (no Draft)
🚫 Caught before execution: [R2] Strategy must begin with a DraftNode. Without a draft model, there are no tokens to verify.

Static checker: 1 valid strategy passed, 2 broken strategies caught


In [5]:
# ── baseline.py ──────────────────────────────────────────────
# Runs HuggingFace spec-1 (batch=1) vs autoregressive baseline.

import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM

DRAFT_MODEL = "facebook/opt-125m"
TARGET_MODEL = "facebook/opt-1.3b"
MAX_NEW_TOKENS = 40
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Loading models… (this takes ~2 min on first run)")

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

target = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, torch_dtype=torch.float16, device_map="auto")
draft = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL, torch_dtype=torch.float16, device_map="auto")

target.eval(); draft.eval()

# Allocate VRAM usage
if DEVICE == "cuda":
    allocated = torch.cuda.memory_allocated()/ 1e9
    reserved = torch.cuda.memory_reserved()/ 1e9
    print(f"VRAM used: {allocated:.2f} GB allocated / {reserved:.2f} GB reserved")

PROMPTS = [
    "The weather today is",
    "In machine learning, a transformer model",
    "The capital of France is",
    "Speculative decoding works by",
    "The best way to learn programming is",
    "Artificial intelligence will change",
    "The most important invention of the 20th century",
    "In a recent study, researchers found that",
]

inputs = tokenizer(PROMPTS, return_tensors="pt", padding=True).to(DEVICE)
print(f"\nRunning on {len(PROMPTS)} prompts, {MAX_NEW_TOKENS} new tokens each")

# ── Autoregressive baseline ───────────────────────────────────
with torch.no_grad():
    t0 = time.time()
    ar_out = target.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    ar_time = time.time() - t0

ar_new = ar_out[:, inputs["input_ids"].shape[1]:]
ar_total = ar_new.numel()
ar_tps = ar_total / ar_time
print(f"\n[AR baseline] {ar_tps:.1f} tokens/s ({ar_total} tokens in {ar_time:.2f}s)")

# ── Speculative decoding — batch=1 loop ───────────────────────
spec_outs = []
t0 = time.time()
for prompt in PROMPTS:
    single = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = target.generate(
            **single,
            assistant_model=draft,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )
    spec_outs.append(out[0, single["input_ids"].shape[1]:])
spec_time = time.time() - t0

spec_total = sum(t.numel() for t in spec_outs)
spec_tps = spec_total / spec_time
print(f"[Spec batch=1] {spec_tps:.1f} tokens/s ({spec_total} tokens in {spec_time:.2f}s)")
print(f"[Speedup] {spec_tps/ar_tps:.2f}x")

# ── Exact-match check ─────────────────────────────────────────
exact = 0
for i, (ar_tok, sp_tok) in enumerate(zip(ar_new, spec_outs)):
    L = min(len(ar_tok), len(sp_tok))
    if (ar_tok[:L] == sp_tok[:L]).all():
        exact += 1
    else:
        for j in range(L):
            if ar_tok[j] != sp_tok[j]:
                print(f" Prompt {i}: first divergence at token {j}")
                break

rate = exact / len(PROMPTS)
print(f"[Exact match ] {rate:.1%} ({exact}/{len(PROMPTS)})")

# ── Summary ───────────────────────────────────────────────────
print("\n" + "="*55)
print("WEEK 1 BASELINE")
print("="*55)
print(f"  Model pair    : {DRAFT_MODEL} → {TARGET_MODEL}")
print(f"  AR throughput : {ar_tps:.1f} tokens/s")
print(f"  Spec-1 throughput: {spec_tps:.1f} tokens/s")
print(f"  Speedup       : {spec_tps/ar_tps:.2f}x")
print(f"  Exact match   : {rate:.1%}")
print(f"  Time AR       : {ar_time:.2f}s")
print(f"  Time Spec-1   : {spec_time:.2f}s")

Device: cuda
Loading models… (this takes ~2 min on first run)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

VRAM used: 3.17 GB allocated / 3.19 GB reserved


model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]


Running on 8 prompts, 40 new tokens each

[AR baseline] 7.6 tokens/s (320 tokens in 41.93s)


Passing `generation_config` together with generation-related arguments=({'min_new_tokens', 'use_cache', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=45) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=45) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_cla

[Spec batch=1] 5.9 tokens/s (320 tokens in 54.37s)
[Speedup] 0.77x
[Exact match ] 100.0% (8/8)

WEEK 1 BASELINE — COPY THESE INTO YOUR REPORT
  Model pair    : facebook/opt-125m → facebook/opt-1.3b
  AR throughput : 7.6 tokens/s
  Spec-1 throughput: 5.9 tokens/s
  Speedup       : 0.77x
  Exact match   : 100.0%
  Time AR       : 41.93s
  Time Spec-1   : 54.37s


In [6]:
# Prints a clean summary of what the static checker caught.
print("=" * 60)
print("SPECSCRIPT STATIC CHECKER — EXECUTION LOG")
print("=" * 60)

tests = [
    ("VALID",  "DRAFT → VERIFY → ACCEPT → REALIGN",  "PASSED", "EQSPEC strategy"),
    ("INVALID", "DRAFT → VERIFY → ACCEPT (no REALIGN)", "CAUGHT", "DSD/BSP bug class"),
    ("INVALID", "VERIFY → ACCEPT → REALIGN (no DRAFT)", "CAUGHT", "Missing draft model"),
]

for kind, graph, result, note in tests:
    icon = "✅" if result == "PASSED" else "🚫"
    print(f"\n {icon} [{kind}] {graph}")
    print(f"     Result : {result} before execution")
    print(f"     Note : {note}")

print("\n" + "=" * 60)
print("The checker enforces invariants I1, I2, and KV-cache alignment")
print("from Zhang et al. (ICLR 2026) at compile time, not runtime.")

SPECSCRIPT STATIC CHECKER — EXECUTION LOG

 ✅ [VALID] DRAFT → VERIFY → ACCEPT → REALIGN
     Result : PASSED before execution
     Note : EQSPEC strategy

 🚫 [INVALID] DRAFT → VERIFY → ACCEPT (no REALIGN)
     Result : CAUGHT before execution
     Note : DSD/BSP bug class

 🚫 [INVALID] VERIFY → ACCEPT → REALIGN (no DRAFT)
     Result : CAUGHT before execution
     Note : Missing draft model

The checker enforces invariants I1, I2, and KV-cache alignment
from Zhang et al. (ICLR 2026) at compile time, not runtime.


In [7]:
# ── specscript/builder.py ─────────────────────────────────────
# The fluent API. This is what makes SpecScript a DSL:
# instead of constructing nodes manually, the user chains
# method calls to declare a strategy in a few lines.

class SpecScript:
    """
    User-facing builder. Chain methods to declare a strategy,
    then call .compile() to validate and get an executor.

    Example — EQSPEC strategy:
        runner = (
            SpecScript()
            .draft(model="facebook/opt-125m", k=5)
            .verify(model="facebook/opt-1.3b")
            .accept()
            .realign()
            .compile()
        )

    Example — EXSPEC strategy (pool instead of realign):
        runner = (
            SpecScript()
            .draft(model="facebook/opt-125m", k=5)
            .verify(model="facebook/opt-1.3b")
            .accept()
            .pool(window_size=8)
            .compile()
        )
    """

    def __init__(self):
        self._graph = SpecGraph()

    def draft(self, model: str, k: int = 5) -> "SpecScript":
        node = DraftNode(node_id=0, draft_model_ref=model, num_draft_tokens=k)
        prev = self._graph.nodes[-1].node_id if self._graph.nodes else None
        added = self._graph.add(node)
        if prev is not None:
            added.inputs = [prev]
        return self

    def verify(self, model: str) -> "SpecScript":
        prev = self._graph.nodes[-1].node_id if self._graph.nodes else None
        node = VerifyNode(node_id=0, target_model_ref=model,
                          inputs=[prev] if prev is not None else [])
        self._graph.add(node)
        return self

    def accept(self, sampling: str = "greedy") -> "SpecScript":
        prev = self._graph.nodes[-1].node_id if self._graph.nodes else None
        node = AcceptNode(node_id=0, sampling=sampling,
                          inputs=[prev] if prev is not None else [])
        self._graph.add(node)
        return self

    def realign(self) -> "SpecScript":
        prev = self._graph.nodes[-1].node_id if self._graph.nodes else None
        node = RealignNode(node_id=0, inputs=[prev] if prev is not None else [])
        self._graph.add(node)
        return self

    def pool(self, window_size: int = 16) -> "SpecScript":
        prev = self._graph.nodes[-1].node_id if self._graph.nodes else None
        node = PoolNode(node_id=0, window_size=window_size,
                        inputs=[prev] if prev is not None else [])
        self._graph.add(node)
        return self

    def compile(self) -> "EQSPECExecutor":
        """Run static checker, then return a ready-to-run executor."""
        check_strategy(self._graph) # raises SpecScriptError if invalid
        draft_node = next(n for n in self._graph.nodes if n.kind == NodeKind.DRAFT)
        verify_node = next(n for n in self._graph.nodes if n.kind == NodeKind.VERIFY)
        print(f"✅ Compiled: {' → '.join(n.kind.name for n in self._graph.nodes)}")
        return EQSPECExecutor(
            draft_model_ref = draft_node.draft_model_ref,
            target_model_ref = verify_node.target_model_ref,
            num_draft_tokens = draft_node.num_draft_tokens,
            graph = self._graph
        )

    def _get_graph(self) -> SpecGraph:
        return self._graph


# ── Quick test ────────────────────────────────────────────────
print("Builder test — EQSPEC strategy:")
g = (SpecScript()
     .draft(model="facebook/opt-125m", k=5)
     .verify(model="facebook/opt-1.3b")
     .accept()
     .realign())
print(f" Graph: {' → '.join(n.kind.name for n in g._get_graph().nodes)}")

print("\nBuilder test — INVALID strategy (will be caught at .compile()):")
try:
    bad = (SpecScript()
           .draft(model="facebook/opt-125m", k=5)
           .verify(model="facebook/opt-1.3b")
           .accept() # missing .realign()
           .compile())
except SpecScriptError as e:
    print(f"🚫 Caught: {e}")

Builder test — EQSPEC strategy:
 Graph: DRAFT → VERIFY → ACCEPT → REALIGN

Builder test — INVALID strategy (will be caught at .compile()):
🚫 Caught: [R1] AcceptNode at position 2 is the last node — no RealignNode follows it. This violates INV-1 (rectangular alignment), INV-2 (position-ID contiguity), and INV-3 (KV-cache correspondence). This is the exact bug class found in BSP, DSD, and Meta's system.


In [8]:
# ── specscript/executor.py ───────
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class EQSPECExecutor:
    draft_model_ref: str
    target_model_ref: str
    num_draft_tokens: int
    graph: SpecGraph

    _tokenizer: object = field(default=None, repr=False)
    _target_model: object = field(default=None, repr=False)
    _draft_model: object = field(default=None, repr=False)
    _device: object = field(default=None, repr=False)

    def _load(self):
        if self._tokenizer is not None:
            return
        print("Loading models for execution…")
        self._tokenizer = AutoTokenizer.from_pretrained(
            self.target_model_ref, padding_side="left")
        self._tokenizer.pad_token = self._tokenizer.eos_token

        self._target_model = AutoModelForCausalLM.from_pretrained(
            self.target_model_ref, torch_dtype=torch.float16,
            device_map="auto")
        self._draft_model = AutoModelForCausalLM.from_pretrained(
            self.draft_model_ref, torch_dtype=torch.float16,
            device_map="auto")
        self._target_model.eval()
        self._draft_model.eval()
        self._device = next(self._target_model.parameters()).device
        print("Models ready.")

    def _pos_ids(self, mask):
        """
        Compute position IDs from attention mask.
        This is Invariant I2 from Zhang et al. (2026):
        position IDs count only content tokens, starting from 0
        at the first non-padding token. Padding tokens get position 0
        but are excluded by the attention mask.
        Without this, OPT assigns wrong positions to content tokens
        after every repad — causing the divergence from the AR baseline.
        """
        pos = mask.long().cumsum(-1) - 1
        pos = pos.clamp(min=0)
        return pos

    def run(self, prompts: List[str],
            max_new_tokens: int = 40,
            batch_size: Optional[int] = None) -> List[str]:
        self._load()
        bs = batch_size or len(prompts)
        all_out = []
        for start in range(0, len(prompts), bs):
            batch = prompts[start : start + bs]
            all_out.extend(self._run_batch(batch, max_new_tokens))
        return all_out

    def _run_batch(self, prompts, max_new_tokens):
        tok = self._tokenizer
        dev = self._device
        K = self.num_draft_tokens
        pad_id = tok.pad_token_id
        eos_id = tok.eos_token_id
        B = len(prompts)

        enc = tok(prompts, return_tensors="pt", padding=True).to(dev)
        ids = enc["input_ids"]
        mask = enc["attention_mask"]

        generated = [[] for _ in range(B)]
        done = [False] * B
        total_new = 0

        while total_new < max_new_tokens and not all(done):
            remaining = max_new_tokens - total_new

            # ── Phase 1: Draft k tokens ───────────────────────
            draft_ids = ids.clone()
            draft_mask = mask.clone()
            draft_toks = []

            for _ in range(min(K, remaining)):
                # position_ids computed from draft_mask
                draft_pos = self._pos_ids(draft_mask)
                with torch.no_grad():
                    d_out = self._draft_model(
                        input_ids = draft_ids,
                        attention_mask = draft_mask,
                        position_ids = draft_pos,
                    )
                next_tok = d_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                draft_toks.append(next_tok)
                draft_ids = torch.cat([draft_ids, next_tok], dim=1)
                new_col = torch.ones(B, 1, dtype=mask.dtype, device=dev)
                draft_mask = torch.cat([draft_mask, new_col], dim=1)

            if not draft_toks:
                break

            k_actual = len(draft_toks)
            draft_mat = torch.cat(draft_toks, dim=1)

            # ── Phase 2: Target verifies full sequence + draft ─
            verify_ids = torch.cat([ids, draft_mat], dim=1)
            verify_mask = torch.cat(
                [mask, torch.ones(B, k_actual, dtype=mask.dtype, device=dev)],
                dim=1)


            verify_pos = self._pos_ids(verify_mask)

            with torch.no_grad():
                t_out = self._target_model(
                    input_ids = verify_ids,
                    attention_mask = verify_mask,
                    position_ids = verify_pos,
                )

            L = ids.shape[1]

            # ── Phase 3: Accept / Reject ───────────────────────
            accepted = []
            bonus = []

            for i in range(B):
                seq_accepted = []
                bonus_tok = None

                for j in range(k_actual):
                    target_pred = t_out.logits[i, L - 1 + j, :].argmax().item()
                    draft_tok = draft_mat[i, j].item()

                    if target_pred == draft_tok:
                        seq_accepted.append(draft_tok)
                    else:
                        bonus_tok = target_pred
                        break

                if bonus_tok is None:
                    bonus_tok = t_out.logits[i, L - 1 + k_actual, :].argmax().item()

                accepted.append(seq_accepted)
                bonus.append(bonus_tok)

                if not done[i]:
                    for tk in seq_accepted:
                        generated[i].append(tk)
                    generated[i].append(bonus_tok)
                    if bonus_tok == eos_id:
                        done[i] = True

            # ── Phase 4: Unpad-Append-Repad (I1 + I2) ─────────
            new_seqs = []
            for i in range(B):
                content_start = (mask[i] == 1).nonzero(as_tuple=True)[0][0].item()
                content = ids[i, content_start:]
                new_toks = torch.tensor(
                    accepted[i] + [bonus[i]],
                    dtype=ids.dtype, device=dev)
                new_seqs.append(torch.cat([content, new_toks]))

            new_L = max(s.shape[0] for s in new_seqs)
            new_ids_list, new_mask_list = [], []
            for seq in new_seqs:
                pad_len = new_L - seq.shape[0]
                new_ids_list.append(torch.cat([
                    torch.full((pad_len,), pad_id, dtype=ids.dtype, device=dev),
                    seq
                ]))
                new_mask_list.append(torch.cat([
                    torch.zeros(pad_len, dtype=mask.dtype, device=dev),
                    torch.ones(seq.shape[0], dtype=mask.dtype, device=dev)
                ]))

            ids  = torch.stack(new_ids_list)
            mask = torch.stack(new_mask_list)
            # I2 is now enforced: next iteration's _pos_ids(mask)
            # will correctly count only content tokens — FIX 3 OF 3
            # is that we do NOT reuse old position_ids here; they are
            # always recomputed fresh from the new mask at the top
            # of the next draft loop and the next verify call.

            min_added = min(len(a) + 1 for a in accepted)
            total_new += min_added

            if any(bonus[i] == eos_id for i in range(B)):
                break

        decoded = [tok.decode(generated[i], skip_special_tokens=True)
                   for i in range(B)]
        return decoded


print("✅ EQSPECExecutor (position_ids fixed) loaded")
print("   I1: rectangular alignment via unpad-append-repad")
print("   I2: position IDs computed from mask on every forward pass")
print("   I3: full sequence recompute each round")

✅ EQSPECExecutor (position_ids fixed) loaded
   I1: rectangular alignment via unpad-append-repad
   I2: position IDs computed from mask on every forward pass
   I3: full sequence recompute each round


In [9]:
import time

PROMPTS_TEST = [
    "The weather today is",
    "In machine learning, a transformer model",
    "The capital of France is",
    "Speculative decoding works by",
    "The best way to learn programming is",
    "Artificial intelligence will change",
]

MAX_NEW = 30

print("Building strategy via SpecScript DSL…")
runner = (
    SpecScript()
    .draft(model="facebook/opt-125m", k=5)
    .verify(model="facebook/opt-1.3b")
    .accept()
    .realign()
    .compile()
)

# ── AR baseline (reuse target model from Cell 4 if still loaded) ──
print("\nRunning AR baseline…")
enc = tokenizer(PROMPTS_TEST, return_tensors="pt", padding=True).to(DEVICE)
with torch.no_grad():
    t0 = time.time()
    ar_out = target.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False)
    ar_time = time.time() - t0

ar_new_toks = ar_out[:, enc["input_ids"].shape[1]:]
ar_decoded = tokenizer.batch_decode(ar_new_toks, skip_special_tokens=True)
ar_tps = ar_new_toks.numel() / ar_time
print(f"AR: {ar_tps:.1f} tokens/s")

# ── SpecScript EQSPEC batch=1 ─────────────────────────────────
print("\nRunning SpecScript EQSPEC (batch=1)…")
t0 = time.time()
spec_decoded_b1 = runner.run(PROMPTS_TEST, max_new_tokens=MAX_NEW, batch_size=1)
spec_time_b1 = time.time() - t0
spec_tps_b1 = (MAX_NEW * len(PROMPTS_TEST)) / spec_time_b1

# ── SpecScript EQSPEC batch=2 ─────────────────────────────────
print("Running SpecScript EQSPEC (batch=2)…")
t0 = time.time()
spec_decoded_b2 = runner.run(PROMPTS_TEST, max_new_tokens=MAX_NEW, batch_size=2)
spec_time_b2 = time.time() - t0
spec_tps_b2 = (MAX_NEW * len(PROMPTS_TEST)) / spec_time_b2

# ── SpecScript EQSPEC batch=4 ─────────────────────────────────
print("Running SpecScript EQSPEC (batch=4)…")
t0 = time.time()
spec_decoded_b4 = runner.run(PROMPTS_TEST, max_new_tokens=MAX_NEW, batch_size=4)
spec_time_b4 = time.time() - t0
spec_tps_b4 = (MAX_NEW * len(PROMPTS_TEST)) / spec_time_b4

# ── Exact-match check ─────────────────────────────────────────
def exact_match(ar_list, spec_list, label):
    exact = sum(1 for a, s in zip(ar_list, spec_list) if a.strip() == s.strip())
    print(f"\n[{label}] Exact match: {exact}/{len(ar_list)} = {exact/len(ar_list):.1%}")
    for i, (a, s) in enumerate(zip(ar_list, spec_list)):
        icon = "✅" if a.strip() == s.strip() else "❌"
        print(f"  {icon} Prompt {i}: AR='{a[:40]}…'  SPEC='{s[:40]}…'")
    return exact / len(ar_list)
# Truncate spec outputs to same length as AR would produce
# The executor may overshoot max_new_tokens in the final round
def truncate_to_match(ar_list, spec_list, tokenizer, max_tokens):
    result = []
    for ar, spec in zip(ar_list, spec_list):
        ar_toks   = tokenizer.encode(ar,   add_special_tokens=False)
        spec_toks = tokenizer.encode(spec, add_special_tokens=False)
        # Truncate spec to same number of tokens as AR produced
        spec_toks = spec_toks[:len(ar_toks)]
        result.append(tokenizer.decode(spec_toks, skip_special_tokens=True))
    return result

tok_for_truncate = runner._tokenizer
spec_decoded_b1 = truncate_to_match(ar_decoded, spec_decoded_b1, tok_for_truncate, MAX_NEW)
spec_decoded_b2 = truncate_to_match(ar_decoded, spec_decoded_b2, tok_for_truncate, MAX_NEW)
spec_decoded_b4 = truncate_to_match(ar_decoded, spec_decoded_b4, tok_for_truncate, MAX_NEW)
rate_b1 = exact_match(ar_decoded, spec_decoded_b1, "Batch=1")
rate_b2 = exact_match(ar_decoded, spec_decoded_b2, "Batch=2")
rate_b4 = exact_match(ar_decoded, spec_decoded_b4, "Batch=4")

print("\n" + "="*55)
print("CORRECTNESS + THROUGHPUT SUMMARY")
print("="*55)
print(f"  AR  throughput    : {ar_tps:.1f} tokens/s")
print(f"  Spec-1 throughput : {spec_tps_b1:.1f} tokens/s   exact={rate_b1:.1%}")
print(f"  Spec-2 throughput : {spec_tps_b2:.1f} tokens/s   exact={rate_b2:.1%}")
print(f"  Spec-4 throughput : {spec_tps_b4:.1f} tokens/s   exact={rate_b4:.1%}")
print(f"  Speedup B=1 vs AR : {spec_tps_b1/ar_tps:.2f}x")
print(f"  Speedup B=2 vs AR : {spec_tps_b2/ar_tps:.2f}x")
print(f"  Speedup B=4 vs AR : {spec_tps_b4/ar_tps:.2f}x")
print("="*55)
print("Model pair: facebook/opt-125m (draft) → facebook/opt-1.3b (target)")
print(f"Hardware  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Draft K   : {runner.num_draft_tokens} tokens per speculation round")

Building strategy via SpecScript DSL…
✅ Static check PASSED — strategy satisfies all synchronization invariants
✅ Compiled: DRAFT → VERIFY → ACCEPT → REALIGN

Running AR baseline…
AR: 318.0 tokens/s

Running SpecScript EQSPEC (batch=1)…
Loading models for execution…


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Models ready.
Running SpecScript EQSPEC (batch=2)…
Running SpecScript EQSPEC (batch=4)…

[Batch=1] Exact match: 6/6 = 100.0%
  ✅ Prompt 0: AR=' going to be a bit of a mixed bag.
It's …'  SPEC=' going to be a bit of a mixed bag.
It's …'
  ✅ Prompt 1: AR=' is a model that transforms a set of dat…'  SPEC=' is a model that transforms a set of dat…'
  ✅ Prompt 2: AR=' Paris.
I know, but I was just wondering…'  SPEC=' Paris.
I know, but I was just wondering…'
  ✅ Prompt 3: AR=' using a set of rules to determine the p…'  SPEC=' using a set of rules to determine the p…'
  ✅ Prompt 4: AR=' to do it.                          …'  SPEC=' to do it.                          …'
  ✅ Prompt 5: AR=' the way we work, study says
Artificial …'  SPEC=' the way we work, study says
Artificial …'

[Batch=2] Exact match: 6/6 = 100.0%
  ✅ Prompt 0: AR=' going to be a bit of a mixed bag.
It's …'  SPEC=' going to be a bit of a mixed bag.
It's …'
  ✅ Prompt 1: AR=' is a model that transforms a set of dat…'  SPEC=' 